# Cadenas de Markov y Tasa de Entropía

Este cuaderno ilustra la tasa de entropía $\bar{H}$ para cadenas de Markov y
el teorema AEP empírico para fuentes con memoria, acompañando el artículo
[Cadenas de Markov y Tasa de Entropía](../../02-teoria-informacion/12-cadenas-de-markov-y-tasa-de-entropia.md).

Solo se usa la biblioteca estándar de Python.

In [ ]:
import math
import random
from collections import defaultdict, Counter

## Utilidades

In [ ]:
def entropy(probs):
    """Entropía de Shannon en bits dado un iterable de probabilidades."""
    return -sum(p * math.log2(p) for p in probs if p > 0)


def stationary_distribution(P, tol=1e-10, max_iter=10000):
    """Distribución estacionaria de la cadena de Markov P por iteración de potencia.
    
    P: lista de listas (matriz de transición, filas suman 1)
    Retorna lista pi con pi[i] = P(estado i en estacionario)
    """
    n = len(P)
    pi = [1.0 / n] * n
    for _ in range(max_iter):
        new_pi = [0.0] * n
        for j in range(n):
            for i in range(n):
                new_pi[j] += pi[i] * P[i][j]
        if max(abs(new_pi[i] - pi[i]) for i in range(n)) < tol:
            return new_pi
        pi = new_pi
    return pi


def entropy_rate(P):
    """Tasa de entropía H_bar = sum_i pi_i * H(fila i de P)."""
    pi = stationary_distribution(P)
    n = len(P)
    h_bar = 0.0
    for i in range(n):
        h_bar += pi[i] * entropy(P[i])
    return h_bar


def sample_markov(P, n_steps, seed=None):
    """Genera una secuencia de n_steps pasos siguiendo la cadena P."""
    if seed is not None:
        random.seed(seed)
    n = len(P)
    state = random.randrange(n)
    seq = [state]
    for _ in range(n_steps - 1):
        r = random.random()
        cumulative = 0.0
        for j, p in enumerate(P[state]):
            cumulative += p
            if r <= cumulative:
                state = j
                break
        seq.append(state)
    return seq


print("Utilidades definidas.")

## Ejemplo 1: Cadena de Markov binaria

Cadena binaria con probabilidad de cambio de estado $p$ (de 0 a 1) y $q$ (de 1 a 0).

In [ ]:
def binary_markov(p, q):
    """Matriz de transición binaria: P[0][1]=p, P[1][0]=q."""
    return [[1 - p, p], [q, 1 - q]]


# Distribución estacionaria analítica: pi_0 = q/(p+q), pi_1 = p/(p+q)
def stationary_binary(p, q):
    return [q / (p + q), p / (p + q)]


# Tasa de entropía analítica
def entropy_rate_binary(p, q):
    pi = stationary_binary(p, q)
    h0 = entropy([1 - p, p])
    h1 = entropy([q, 1 - q])
    return pi[0] * h0 + pi[1] * h1


print("Cadena de Markov binaria: comparación analítica vs. iteración de potencia")
print(f"{'p':>5} {'q':>5} | {'pi_0 anal':>10} {'pi_0 iter':>10} | {'H_bar anal':>10} {'H_bar iter':>10}")
print("-" * 60)

test_cases = [(0.1, 0.2), (0.3, 0.3), (0.5, 0.5), (0.1, 0.9), (0.4, 0.1)]
for p, q in test_cases:
    P = binary_markov(p, q)
    pi_iter = stationary_distribution(P)
    pi_anal = stationary_binary(p, q)
    h_anal = entropy_rate_binary(p, q)
    h_iter = entropy_rate(P)
    print(f"{p:>5.1f} {q:>5.1f} | {pi_anal[0]:>10.5f} {pi_iter[0]:>10.5f} | {h_anal:>10.5f} {h_iter:>10.5f}")

## Ejemplo 2: Convergencia empírica de H/n a H_bar

Para una secuencia de longitud $n$, estimamos $-\frac{1}{n}\log_2 P(x_1,\ldots,x_n)$
y comprobamos que converge a $\bar{H}$.

In [ ]:
def log_prob_markov(seq, P):
    """Log probabilidad de la secuencia bajo la cadena P (estado inicial uniforme)."""
    n_states = len(P)
    log_p = math.log2(1.0 / n_states)  # prior uniforme para el primer estado
    for t in range(1, len(seq)):
        prob_trans = P[seq[t-1]][seq[t]]
        if prob_trans > 0:
            log_p += math.log2(prob_trans)
        else:
            return float('-inf')
    return log_p


# Cadena de Markov 3-estados (texto simplificado: vocales, consonantes, espacios)
P_texto = [
    [0.1, 0.7, 0.2],   # desde vocal
    [0.5, 0.2, 0.3],   # desde consonante
    [0.6, 0.4, 0.0],   # desde espacio
]

h_bar_teorica = entropy_rate(P_texto)
print(f"Tasa de entropía teórica: {h_bar_teorica:.5f} bits/símbolo")
print()
print("Convergencia de -log2(P(x^n))/n")
print(f"{'n':>7} | {'estimado':>10} | {'H_bar':>10} | {'error':>8}")
print("-" * 44)

random.seed(42)
for n in [100, 500, 1000, 5000, 10000, 50000]:
    seq = sample_markov(P_texto, n, seed=None)
    lp = log_prob_markov(seq, P_texto)
    estimated = -lp / n
    print(f"{n:>7} | {estimated:>10.5f} | {h_bar_teorica:>10.5f} | {abs(estimated - h_bar_teorica):>8.5f}")

## Ejemplo 3: Tasa de entropía de orden k en texto real

Estimamos $\bar{H}_k = H(X_n \mid X_{n-1},\ldots,X_{n-k})$ para distintos k
usando frecuencias de k-gramas en un texto.

In [ ]:
def estimate_entropy_rate_order_k(text, k):
    """Estima H_bar_k = H(X_n | X_{n-k},...,X_{n-1}) por frecuencias de (k+1)-gramas."""
    # Contar (k+1)-gramas y k-gramas
    kp1_grams = Counter(text[i:i+k+1] for i in range(len(text) - k))
    k_grams = Counter(text[i:i+k] for i in range(len(text) - k + 1))
    
    # H_bar_k = sum_{context} P(context) * H(next | context)
    total = sum(k_grams.values())
    h_bar = 0.0
    
    for context, count_ctx in k_grams.items():
        # Buscar todos los (k+1)-gramas con este prefijo
        continuations = {g[k:]: c for g, c in kp1_grams.items() if g[:k] == context}
        if not continuations:
            continue
        total_ctx = sum(continuations.values())
        probs = [c / total_ctx for c in continuations.values()]
        h_ctx = entropy(probs)
        h_bar += (count_ctx / total) * h_ctx
    
    return h_bar


# Texto de prueba en español (primeras líneas del Quijote, simplificado)
texto = (
    "en un lugar de la mancha de cuyo nombre no quiero acordarme no ha mucho tiempo "
    "que vivia un hidalgo de los de lanza en astillero adarga antigua rocin flaco y "
    "galgo corredor una olla de algo mas vaca que carnero salpicon las mas noches "
    "duelos y quebrantos los sabados lentejas los viernes algun palomino de annadidura "
    "los domingos consumian las tres partes de su hacienda el resto della concluian "
    "sayo de velarte calzas de velludo para las fiestas con sus pantuflos de lo mesmo "
) * 5  # Repetimos para tener más datos

print(f"Texto de prueba: {len(texto)} caracteres\n")
print("Tasa de entropía de orden k (estimada empíricamente):")
print(f"{'k':>3} | {'H_bar_k':>8} bits/car")
print("-" * 22)

h_1 = estimate_entropy_rate_order_k(texto, 1)
for k in range(1, 6):
    h_k = estimate_entropy_rate_order_k(texto, k)
    print(f"{k:>3} | {h_k:>8.4f}")
    
# Entropía de orden 0 (sin contexto)
freq = Counter(texto)
total = len(texto)
h0 = entropy([c/total for c in freq.values()])
print(f"{'0':>3} | {h0:>8.4f}  (sin contexto)")

## Ejemplo 4: Comparación de H(X_1) vs H_bar

La memoria siempre reduce la tasa de entropía: $\bar{H} \leq H(X_1)$.

In [ ]:
# Cadenas con distintos grados de memoria
print("Comparación H(X_1) vs H_bar para distintos modelos de Markov binarios")
print(f"{'p':>5} {'q':>5} | {'H(X_1)':>8} | {'H_bar':>8} | {'reducción':>10}")
print("-" * 48)

for p, q in [(0.01, 0.01), (0.1, 0.1), (0.3, 0.3), (0.5, 0.5),
             (0.1, 0.9), (0.4, 0.6), (0.01, 0.99)]:
    pi = stationary_binary(p, q)
    h_x1 = entropy(pi)  # entropía marginal
    h_bar = entropy_rate_binary(p, q)
    reduccion = (h_x1 - h_bar) / h_x1 * 100 if h_x1 > 0 else 0
    print(f"{p:>5.2f} {q:>5.2f} | {h_x1:>8.4f} | {h_bar:>8.4f} | {reduccion:>9.1f}%")

print("\nObservación: la reducción es mayor cuando p y q son pequeños")
print("(la cadena 'persiste' más en cada estado → más predecible)")

## Ideas clave

- La **tasa de entropía** $\bar{H}$ es la incertidumbre por símbolo en el límite
  asintótico; siempre es $\leq H(X_1)$, con igualdad si la cadena es i.i.d.
- Para cadenas de Markov ergódicas: $\bar{H} = \sum_i \pi_i H(\text{fila}_i)$.
- El AEP se verifica empíricamente: $-\log_2 P(x^n)/n \to \bar{H}$ conforme $n\to\infty$.
- El contexto k-grama reduce progresivamente la entropía estimada; el límite es $\bar{H}$.
- La compresión óptima de texto real requiere explotar la memoria del lenguaje.

## Ejercicios

1. Para la cadena binaria simétrica con $p = q$, demuestra analíticamente que
   $\bar{H} = H_b(p)$ donde $H_b$ es la entropía binaria. Verifica numéricamente.
2. ¿Cuál es la cadena de Markov binaria (con $p + q = 0.5$ fijo) que maximiza $\bar{H}$?
3. Implementa la estimación de $\bar{H}_k$ por máxima verosimilitud y compara
   con el modelo de proceso real para la cadena 3-estados del Ejemplo 2.
4. Genera una secuencia de 10.000 símbolos con la cadena P_texto y compara
   la longitud de compresión con zlib (si está disponible) con el límite teórico
   $n \cdot \bar{H}$ bits.